# 01 — Data Collection

This notebook collects all raw data needed for NBA player injury prediction.

**Data sources:**
1. **elap733/NBA-Injuries-Analysis** — Pre-scraped injury transaction data from prosportstransactions.com (2010-2019)
2. **nba_api** — Official NBA stats API for player statistics, bio data, tracking stats, team schedules, and hustle stats

**Pipeline role:** This is the first notebook. It produces raw CSV files in `data/raw/` that are consumed by `02_data_cleaning.ipynb`.

**Caching:** Every API call checks for an existing CSV first (`skip-if-exists`). Re-running this notebook is safe and fast if the data is already cached.

In [1]:
import sys
sys.path.append('..')

import os
import time
import pandas as pd
import numpy as np
from pathlib import Path

from src.config import (
    FIRST_SEASON, LAST_SEASON,
    TRACKING_DATA_START,
    RAW_ELAP_DIR, RAW_NBA_API_DIR
)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [2]:
# Ensure output directories exist
Path(f'../{RAW_ELAP_DIR}').mkdir(parents=True, exist_ok=True)
Path(f'../{RAW_NBA_API_DIR}').mkdir(parents=True, exist_ok=True)

print(f"Season range: {FIRST_SEASON} to {LAST_SEASON}")
print(f"Tracking data available from: {TRACKING_DATA_START}")

Season range: 2010 to 2019
Tracking data available from: 2013


In [3]:
def safe_api_call(endpoint_class, max_retries=3, **kwargs):
    """
    Call nba_api endpoint with rate limiting and retry logic.

    Parameters
    ----------
    endpoint_class : class
        nba_api endpoint class to instantiate
    max_retries : int
        Maximum number of retry attempts
    **kwargs : dict
        Arguments to pass to the endpoint

    Returns
    -------
    pd.DataFrame
        First DataFrame from the endpoint response
    """
    time.sleep(1.5)  # Rate limiting

    for attempt in range(max_retries):
        try:
            result = endpoint_class(**kwargs)
            dfs = result.get_data_frames()
            if dfs:
                return dfs[0]
            return pd.DataFrame()
        except Exception as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                wait_time = 5 * (attempt + 1)
                print(f"  Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"  All retries failed, returning empty DataFrame")
                return pd.DataFrame()

---
## Section 1: Download & Load elap733 Injury Data

The [elap733/NBA-Injuries-Analysis](https://github.com/elap733/NBA-Injuries-Analysis) repository contains pre-scraped injury transaction data from prosportstransactions.com (2010-2019). The cell below auto-downloads these CSVs from GitHub on first run (skip-if-exists), so no manual setup is needed.

**Why we need this:** This is the source of our **target variable** — games missed due to injury. Each row in `missed_games` represents one game a player missed. We aggregate these counts per player-season to build `games_missed` (and ultimately `games_missed_next_season` as our prediction target).

The supplementary files (injury transactions, historical player stats, team schedules) provide additional context for cross-referencing and validation.

**Files in `data/raw/elap733/`:**
- `missed_games_2010_2019.csv` — Games missed per injury event **(target variable source)**
- `injury_transactions_2010_2019.csv` — Inactive Reserve List (IRL) transaction records
- `player_stats_1994_2019.csv` — Historical player stats (Basketball Reference)
- `team_schedules_2010_2020.csv` — Team game schedules

In [ ]:
import urllib.request

# Download elap733 raw CSVs from GitHub if not already present locally.
# Source: https://github.com/elap733/NBA-Injuries-Analysis
ELAP_GITHUB_BASE = "https://raw.githubusercontent.com/elap733/NBA-Injuries-Analysis/master/data/01_raw"

ELAP_FILES = {
    "missed_games_2010_2019.csv": f"{ELAP_GITHUB_BASE}/prosportstransactions_scrape_missedgames_2010_2019.csv",
    "injury_transactions_2010_2019.csv": f"{ELAP_GITHUB_BASE}/prosportstransactions_scrape_IRL_2010_2019.csv",
    "player_stats_1994_2019.csv": f"{ELAP_GITHUB_BASE}/player_stats_1994_2019.csv",
    "team_schedules_2010_2020.csv": f"{ELAP_GITHUB_BASE}/all_teams_schedule_2010_2020.csv",
}

for local_name, url in ELAP_FILES.items():
    dest = f"../{RAW_ELAP_DIR}/{local_name}"
    if os.path.exists(dest):
        print(f"[SKIP] {local_name}: Already exists")
    else:
        print(f"[FETCH] {local_name}...", end=" ")
        urllib.request.urlretrieve(url, dest)
        print("Done")

print("\nAll elap733 files ready.")

In [4]:
# Load missed games data — OUR TARGET VARIABLE SOURCE
df_missed_games = pd.read_csv(f'../{RAW_ELAP_DIR}/missed_games_2010_2019.csv', index_col=0)

print("=" * 60)
print("MISSED GAMES DATA (Target Variable Source)")
print("=" * 60)
print(f"Shape: {df_missed_games.shape}")
print(f"\nColumns: {list(df_missed_games.columns)}")
print(f"\nFirst 5 rows:")
df_missed_games.head()

MISSED GAMES DATA (Target Variable Source)
Shape: (11531, 5)

Columns: ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']

First 5 rows:


,Date,Team,Acquired,Relinquished,Notes
0,2010-08-03,Clippers,NaN,Craig Smith,arthroscopic surgery on right knee (out indefi...
1,2010-08-13,Mavericks,NaN,Rodrigue Beaubois,surgery on left foot to repair broken fifth me...
2,2010-08-14,Warriors,NaN,Ekpe Udoh,surgery on left wrist (out indefinitely)
3,2010-09-21,Raptors,NaN,Ed Davis (a),arthroscopic surgery on right kene to repair t...
4,2010-09-21,Thunder,NaN,Nenad Krstic,surgery on right hand to repair broken finger ...


In [5]:
# Explore missed games data structure
print("Data types:")
print(df_missed_games.dtypes)
print(f"\nMissing values:")
print(df_missed_games.isnull().sum())

Data types:
Date            object
Team            object
Acquired        object
Relinquished    object
Notes           object
dtype: object

Missing values:
Date               0
Team               2
Acquired        9328
Relinquished    2203
Notes              0
dtype: int64


The `Acquired` column is mostly NaN because most rows represent players being **placed on** the Injured List (`Relinquished`), not returning from it.

In [6]:
# Load injury transactions data
df_injury_transactions = pd.read_csv(f'../{RAW_ELAP_DIR}/injury_transactions_2010_2019.csv', index_col=0)

print("=" * 60)
print("INJURY TRANSACTIONS DATA")
print("=" * 60)
print(f"Shape: {df_injury_transactions.shape}")
print(f"Columns: {list(df_injury_transactions.columns)}")
df_injury_transactions.head()

INJURY TRANSACTIONS DATA
Shape: (14135, 5)
Columns: ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']


,Date,Team,Acquired,Relinquished,Notes
0,2010-10-26,Blazers,NaN,Elliot Williams,placed on IL
1,2010-10-26,Blazers,NaN,Greg Oden,placed on IL with left knee injury (out for se...
2,2010-10-26,Blazers,NaN,Joel Przybilla,placed on IL placed on IL recovering from surg...
3,2010-10-26,Celtics,NaN,Avery Bradley,placed on IL recovering from surgery on left a...
4,2010-10-26,Celtics,NaN,Kendrick Perkins,placed on IL recovering from surgery on right ...


In [7]:
# Load elap733 player stats (for reference/comparison with nba_api stats)
df_elap_stats = pd.read_csv(f'../{RAW_ELAP_DIR}/player_stats_1994_2019.csv', index_col=0)

print("=" * 60)
print("ELAP733 PLAYER STATS DATA")
print("=" * 60)
print(f"Shape: {df_elap_stats.shape}")
print(f"Columns: {list(df_elap_stats.columns)}")

ELAP733 PLAYER STATS DATA
Shape: (19786, 31)
Columns: ['Year', 'Season', 'Player', 'Pos', 'Age', 'Tm', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']


In [8]:
# Load elap733 team schedules
df_elap_schedules = pd.read_csv(f'../{RAW_ELAP_DIR}/team_schedules_2010_2020.csv', index_col=0)

print("=" * 60)
print("ELAP733 TEAM SCHEDULES DATA")
print("=" * 60)
print(f"Shape: {df_elap_schedules.shape}")
print(f"Columns: {list(df_elap_schedules.columns)}")

ELAP733 TEAM SCHEDULES DATA
Shape: (28240, 8)
Columns: ['Team', 'Year', 'Season', 'Game_num', 'Date', 'Away_flag', 'Opponent', 'OT_flag']


### Section 1 Summary

Loaded 4 CSV files from the elap733 repository:
- **missed_games** — 11,531 rows, our target variable source
- **injury_transactions** — 14,135 IRL transaction records
- **player_stats** — 19,786 rows of historical stats (1994-2019)
- **team_schedules** — 28,240 team-game records (2010-2020)

---
## Section 2: Pull Player Season Stats via nba_api

**Why we need this:** Per-game player statistics (points, rebounds, assists, minutes, usage rate, etc.) are our primary feature set. Higher workload metrics (minutes, usage) may be associated with greater injury risk.

**Endpoint:** `LeagueDashPlayerStats` — returns per-game averages for all players in a given season.

**Output:** `data/raw/nba_api/player_stats_{season}.csv` for each season (2010-2019, 10 files)

In [9]:
from nba_api.stats.endpoints import leaguedashplayerstats

print("Pulling player season stats from nba_api...")
print(f"Seasons: {FIRST_SEASON}-{str(FIRST_SEASON+1)[-2:]} through {LAST_SEASON}-{str(LAST_SEASON+1)[-2:]}")
print("=" * 60)

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"../{RAW_NBA_API_DIR}/player_stats_{season}.csv"

    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists at {filepath}")
        continue

    print(f"[FETCH] {season_str}...", end=" ")

    df = safe_api_call(
        leaguedashplayerstats.LeagueDashPlayerStats,
        season=season_str,
        per_mode_detailed='PerGame'
    )

    if not df.empty:
        df['SEASON'] = season_str
        df['SEASON_START_YEAR'] = season
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} players")
    else:
        print("No data returned")

print("\nDone with player season stats!")

Pulling player season stats from nba_api...
Seasons: 2010-11 through 2019-20
[SKIP] 2010-11: Already exists at ../data/raw/nba_api/player_stats_2010.csv
[SKIP] 2011-12: Already exists at ../data/raw/nba_api/player_stats_2011.csv
[SKIP] 2012-13: Already exists at ../data/raw/nba_api/player_stats_2012.csv
[SKIP] 2013-14: Already exists at ../data/raw/nba_api/player_stats_2013.csv
[SKIP] 2014-15: Already exists at ../data/raw/nba_api/player_stats_2014.csv
[SKIP] 2015-16: Already exists at ../data/raw/nba_api/player_stats_2015.csv
[SKIP] 2016-17: Already exists at ../data/raw/nba_api/player_stats_2016.csv
[SKIP] 2017-18: Already exists at ../data/raw/nba_api/player_stats_2017.csv
[SKIP] 2018-19: Already exists at ../data/raw/nba_api/player_stats_2018.csv
[SKIP] 2019-20: Already exists at ../data/raw/nba_api/player_stats_2019.csv

Done with player season stats!


In [10]:
# Verify player stats files
print("Player stats files:")
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/player_stats_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  player_stats_{season}.csv: {df.shape[0]} players, {df.shape[1]} columns")
    else:
        print(f"  player_stats_{season}.csv: NOT FOUND")

Player stats files:
  player_stats_2010.csv: 452 players, 69 columns
  player_stats_2011.csv: 478 players, 69 columns
  player_stats_2012.csv: 469 players, 69 columns
  player_stats_2013.csv: 482 players, 69 columns
  player_stats_2014.csv: 492 players, 69 columns
  player_stats_2015.csv: 476 players, 69 columns
  player_stats_2016.csv: 486 players, 69 columns
  player_stats_2017.csv: 540 players, 69 columns
  player_stats_2018.csv: 530 players, 69 columns
  player_stats_2019.csv: 529 players, 69 columns


---
## Section 3: Pull Player Bio Data via nba_api

**Why we need this:** Player demographics (height, weight, age, draft position, years of experience) are important control variables and potential injury risk factors. Older, heavier players or those with more NBA seasons may face different injury profiles.

**Endpoint:** `LeagueDashPlayerBioStats` — returns biographical and body measurement data for all players in a season.

**Output:** `data/raw/nba_api/player_bio_{season}.csv` for each season (2010-2019, 10 files)

In [11]:
from nba_api.stats.endpoints import leaguedashplayerbiostats

print("Pulling player bio stats from nba_api...")
print("=" * 60)

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"../{RAW_NBA_API_DIR}/player_bio_{season}.csv"

    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists")
        continue

    print(f"[FETCH] {season_str}...", end=" ")

    df = safe_api_call(
        leaguedashplayerbiostats.LeagueDashPlayerBioStats,
        season=season_str,
        per_mode_simple='PerGame'
    )

    if not df.empty:
        df['SEASON'] = season_str
        df['SEASON_START_YEAR'] = season
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} players")
    else:
        print("No data returned")

print("\nDone with player bio stats!")

Pulling player bio stats from nba_api...
[SKIP] 2010-11: Already exists
[SKIP] 2011-12: Already exists
[SKIP] 2012-13: Already exists
[SKIP] 2013-14: Already exists
[SKIP] 2014-15: Already exists
[SKIP] 2015-16: Already exists
[SKIP] 2016-17: Already exists
[SKIP] 2017-18: Already exists
[SKIP] 2018-19: Already exists
[SKIP] 2019-20: Already exists

Done with player bio stats!


In [12]:
# Verify player bio files
print("Player bio files:")
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/player_bio_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  player_bio_{season}.csv: {df.shape[0]} players, {df.shape[1]} columns")
    else:
        print(f"  player_bio_{season}.csv: NOT FOUND")

Player bio files:
  player_bio_2010.csv: 452 players, 25 columns
  player_bio_2011.csv: 478 players, 25 columns
  player_bio_2012.csv: 469 players, 25 columns
  player_bio_2013.csv: 482 players, 25 columns
  player_bio_2014.csv: 492 players, 25 columns
  player_bio_2015.csv: 476 players, 25 columns
  player_bio_2016.csv: 486 players, 25 columns
  player_bio_2017.csv: 540 players, 25 columns
  player_bio_2018.csv: 530 players, 25 columns
  player_bio_2019.csv: 529 players, 25 columns


---
## Section 4: Pull Player Tracking Stats via nba_api

**Why we need this:** Player tracking data (speed, distance traveled per game) captures physical workload that traditional box-score stats miss. Players who run more miles per game or play at higher average speeds may be at greater injury risk due to cumulative physical stress. These workload-based features are key predictors in sports injury research.

**Availability:** NBA player tracking (Second Spectrum / SportVU) data is only available from the **2013-14 season onward** (`TRACKING_DATA_START=2013`). This gives us 7 seasons of tracking data (2013-2019).

**Endpoint:** `LeagueDashPtStats` with `PtMeasureType='SpeedDistance'` — returns per-player speed and distance metrics.

**Key fields:** `DIST_MILES`, `DIST_MILES_OFF`, `DIST_MILES_DEF`, `AVG_SPEED`, `AVG_SPEED_OFF`, `AVG_SPEED_DEF`

**Output:** `data/raw/nba_api/tracking_stats_{season}.csv` for seasons 2013-2019 (7 files)

In [13]:
from nba_api.stats.endpoints import leaguedashptstats

print("Pulling player tracking stats (speed/distance) from nba_api...")
print(f"Seasons: {TRACKING_DATA_START}-{str(TRACKING_DATA_START+1)[-2:]} through {LAST_SEASON}-{str(LAST_SEASON+1)[-2:]}")
print("=" * 60)

for season in range(TRACKING_DATA_START, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"../{RAW_NBA_API_DIR}/tracking_stats_{season}.csv"

    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists at {filepath}")
        continue

    print(f"[FETCH] {season_str}...", end=" ")

    df = safe_api_call(
        leaguedashptstats.LeagueDashPtStats,
        season=season_str,
        per_mode_simple='PerGame',
        pt_measure_type='SpeedDistance'
    )

    if not df.empty:
        df['SEASON'] = season_str
        df['SEASON_START_YEAR'] = season
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} players")
    else:
        print("No data returned")

print("\nDone with player tracking stats!")

Pulling player tracking stats (speed/distance) from nba_api...
Seasons: 2013-14 through 2019-20
[SKIP] 2013-14: Already exists at ../data/raw/nba_api/tracking_stats_2013.csv
[SKIP] 2014-15: Already exists at ../data/raw/nba_api/tracking_stats_2014.csv
[SKIP] 2015-16: Already exists at ../data/raw/nba_api/tracking_stats_2015.csv
[SKIP] 2016-17: Already exists at ../data/raw/nba_api/tracking_stats_2016.csv
[SKIP] 2017-18: Already exists at ../data/raw/nba_api/tracking_stats_2017.csv
[SKIP] 2018-19: Already exists at ../data/raw/nba_api/tracking_stats_2018.csv
[SKIP] 2019-20: Already exists at ../data/raw/nba_api/tracking_stats_2019.csv

Done with player tracking stats!


In [14]:
# Verify tracking stats files
print("Tracking stats files:")
for season in range(TRACKING_DATA_START, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/tracking_stats_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  tracking_stats_{season}.csv: {df.shape[0]} players, {df.shape[1]} columns")
    else:
        print(f"  tracking_stats_{season}.csv: NOT FOUND")

Tracking stats files:
  tracking_stats_2013.csv: 482 players, 18 columns
  tracking_stats_2014.csv: 492 players, 18 columns
  tracking_stats_2015.csv: 476 players, 18 columns
  tracking_stats_2016.csv: 486 players, 18 columns
  tracking_stats_2017.csv: 540 players, 18 columns
  tracking_stats_2018.csv: 530 players, 18 columns
  tracking_stats_2019.csv: 529 players, 18 columns


---
## Section 5: Pull Team Schedule Data via nba_api

**Why we need this:** Team game schedules let us compute **back-to-back games** — a well-established risk factor for NBA injuries. Players who compete on consecutive days have less recovery time, increasing soft-tissue injury risk. We also use schedules to validate game counts and cross-reference with the elap733 schedule data.

**Endpoint:** `TeamGameLog` — returns all regular-season games for a given team and season.

**Output:** `data/raw/nba_api/team_schedules_{season}.csv` for each season (2010-2019, 10 files)

> **Note:** This section makes ~300 API calls (30 teams × 10 seasons) with rate limiting, so it is the slowest collection step. All data is cached after the first run.

In [15]:
from nba_api.stats.endpoints import teamgamelog
from nba_api.stats.static import teams

# Get all NBA teams
nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} NBA teams")

Found 30 NBA teams


In [16]:
print("Pulling team schedules from nba_api...")
print("=" * 60)

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"../{RAW_NBA_API_DIR}/team_schedules_{season}.csv"

    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists")
        continue

    print(f"[FETCH] {season_str}...")

    all_games = []
    for team in nba_teams:
        team_id = team['id']
        team_abbrev = team['abbreviation']

        df = safe_api_call(
            teamgamelog.TeamGameLog,
            team_id=team_id,
            season=season_str,
            season_type_all_star='Regular Season'
        )

        if not df.empty:
            df['TEAM_ABBREVIATION'] = team_abbrev
            all_games.append(df)
            print(f"  {team_abbrev}: {len(df)} games", end="  ")
            if len(all_games) % 5 == 0:
                print()

    print()

    if all_games:
        df_season = pd.concat(all_games, ignore_index=True)
        df_season['SEASON'] = season_str
        df_season['SEASON_START_YEAR'] = season
        df_season.to_csv(filepath, index=False)
        print(f"  Saved {len(df_season)} total games for {season_str}")
    else:
        print(f"  No games found for {season_str}")

print("\nDone with team schedules!")

Pulling team schedules from nba_api...
[SKIP] 2010-11: Already exists
[SKIP] 2011-12: Already exists
[SKIP] 2012-13: Already exists
[SKIP] 2013-14: Already exists
[SKIP] 2014-15: Already exists
[SKIP] 2015-16: Already exists
[SKIP] 2016-17: Already exists
[SKIP] 2017-18: Already exists
[SKIP] 2018-19: Already exists
[SKIP] 2019-20: Already exists

Done with team schedules!


In [17]:
# Verify team schedule files
print("Team schedule files:")
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/team_schedules_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  team_schedules_{season}.csv: {df.shape[0]} games, {df.shape[1]} columns")
    else:
        print(f"  team_schedules_{season}.csv: NOT FOUND")

Team schedule files:
  team_schedules_2010.csv: 2460 games, 30 columns
  team_schedules_2011.csv: 1980 games, 30 columns
  team_schedules_2012.csv: 2460 games, 30 columns
  team_schedules_2013.csv: 2460 games, 30 columns
  team_schedules_2014.csv: 2460 games, 30 columns
  team_schedules_2015.csv: 2460 games, 30 columns
  team_schedules_2016.csv: 2460 games, 30 columns
  team_schedules_2017.csv: 2460 games, 30 columns
  team_schedules_2018.csv: 2460 games, 30 columns
  team_schedules_2019.csv: 2118 games, 30 columns


---
## Section 6: Hustle Stats

**Why we need this:** Hustle stats (contested shots, deflections, charges drawn, loose balls recovered, box outs) capture defensive effort and physical engagement that standard stats miss. Players with high hustle metrics are more physically involved in plays, which could correlate with injury risk.

**Availability:** NBA hustle stats are only available from the **2016-17 season onward**, giving us 4 seasons of data (2016-2019).

**Endpoint:** `LeagueHustleStatsPlayer` — returns per-player hustle metrics for a given season.

**Output:** `data/raw/nba_api/hustle_stats_2016_2019.csv` (single combined file, 4 seasons)

In [18]:
from nba_api.stats.endpoints import leaguehustlestatsplayer

HUSTLE_START = 2016
hustle_filepath = f"../{RAW_NBA_API_DIR}/hustle_stats_{HUSTLE_START}_{LAST_SEASON}.csv"

if os.path.exists(hustle_filepath):
    print(f"[SKIP] Hustle stats file already exists: {hustle_filepath}")
    df_hustle = pd.read_csv(hustle_filepath)
    print(f"Shape: {df_hustle.shape}")
    print(f"Seasons: {sorted(df_hustle['SEASON'].unique())}")
    print(f"Columns: {list(df_hustle.columns)}")
else:
    print("Pulling hustle stats from nba_api...")
    print("=" * 60)

    all_hustle = []
    for season in range(HUSTLE_START, LAST_SEASON + 1):
        season_str = f"{season}-{str(season+1)[-2:]}"
        print(f"[FETCH] {season_str}...", end=" ")

        df = safe_api_call(
            leaguehustlestatsplayer.LeagueHustleStatsPlayer,
            season=season_str,
            per_mode_time='PerGame'
        )

        if not df.empty:
            df['SEASON'] = season_str
            df['SEASON_START_YEAR'] = season
            all_hustle.append(df)
            print(f"Saved {len(df)} players")
        else:
            print("No data returned")

    if all_hustle:
        df_hustle = pd.concat(all_hustle, ignore_index=True)
        df_hustle.to_csv(hustle_filepath, index=False)
        print(f"\nSaved combined hustle stats: {df_hustle.shape}")
    else:
        print("No hustle data collected")

print("\nDone with hustle stats!")

[SKIP] Hustle stats file already exists: ../data/raw/nba_api/hustle_stats_2016_2019.csv
Shape: (2071, 30)
Seasons: ['2016-17', '2017-18', '2018-19', '2019-20']
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'G', 'MIN', 'CONTESTED_SHOTS', 'CONTESTED_SHOTS_2PT', 'CONTESTED_SHOTS_3PT', 'DEFLECTIONS', 'CHARGES_DRAWN', 'SCREEN_ASSISTS', 'SCREEN_AST_PTS', 'OFF_LOOSE_BALLS_RECOVERED', 'DEF_LOOSE_BALLS_RECOVERED', 'LOOSE_BALLS_RECOVERED', 'PCT_LOOSE_BALLS_RECOVERED_OFF', 'PCT_LOOSE_BALLS_RECOVERED_DEF', 'OFF_BOXOUTS', 'DEF_BOXOUTS', 'BOX_OUT_PLAYER_TEAM_REBS', 'BOX_OUT_PLAYER_REBS', 'BOX_OUTS', 'PCT_BOX_OUTS_OFF', 'PCT_BOX_OUTS_DEF', 'PCT_BOX_OUTS_TEAM_REB', 'PCT_BOX_OUTS_REB', 'SEASON', 'SEASON_START_YEAR']

Done with hustle stats!


---
## Data Validation

Verify that all expected raw data files exist, have reasonable row/column counts, and contain no empty files.

In [19]:
# Define expected files and their minimum expected properties
expected_files = {
    # elap733 files
    f'../{RAW_ELAP_DIR}/missed_games_2010_2019.csv':        {'min_rows': 10000, 'min_cols': 5,  'label': 'elap733 missed games'},
    f'../{RAW_ELAP_DIR}/injury_transactions_2010_2019.csv': {'min_rows': 10000, 'min_cols': 5,  'label': 'elap733 injury txns'},
    f'../{RAW_ELAP_DIR}/player_stats_1994_2019.csv':        {'min_rows': 15000, 'min_cols': 30, 'label': 'elap733 player stats'},
    f'../{RAW_ELAP_DIR}/team_schedules_2010_2020.csv':      {'min_rows': 25000, 'min_cols': 8,  'label': 'elap733 team schedules'},
}

# nba_api player stats (10 seasons)
for s in range(FIRST_SEASON, LAST_SEASON + 1):
    expected_files[f'../{RAW_NBA_API_DIR}/player_stats_{s}.csv'] = {'min_rows': 400, 'min_cols': 60, 'label': f'player_stats_{s}'}

# nba_api player bio (10 seasons)
for s in range(FIRST_SEASON, LAST_SEASON + 1):
    expected_files[f'../{RAW_NBA_API_DIR}/player_bio_{s}.csv'] = {'min_rows': 400, 'min_cols': 20, 'label': f'player_bio_{s}'}

# nba_api tracking stats (7 seasons, 2013-2019)
for s in range(TRACKING_DATA_START, LAST_SEASON + 1):
    expected_files[f'../{RAW_NBA_API_DIR}/tracking_stats_{s}.csv'] = {'min_rows': 400, 'min_cols': 15, 'label': f'tracking_stats_{s}'}

# nba_api team schedules (10 seasons)
for s in range(FIRST_SEASON, LAST_SEASON + 1):
    expected_files[f'../{RAW_NBA_API_DIR}/team_schedules_{s}.csv'] = {'min_rows': 1900, 'min_cols': 25, 'label': f'team_schedules_{s}'}

# Hustle stats (single combined file)
expected_files[f'../{RAW_NBA_API_DIR}/hustle_stats_2016_2019.csv'] = {'min_rows': 1500, 'min_cols': 25, 'label': 'hustle_stats_2016_2019'}

# Run validation
print("=" * 80)
print("DATA VALIDATION")
print("=" * 80)

results = []
all_passed = True

for filepath, spec in expected_files.items():
    label = spec['label']
    exists = os.path.exists(filepath)
    if exists:
        df = pd.read_csv(filepath)
        rows, cols = df.shape
        row_ok = rows >= spec['min_rows']
        col_ok = cols >= spec['min_cols']
        status = "PASS" if (row_ok and col_ok) else "WARN"
        if status == "WARN":
            all_passed = False
        results.append({'File': label, 'Exists': True, 'Rows': rows, 'Cols': cols, 'Status': status})
    else:
        all_passed = False
        results.append({'File': label, 'Exists': False, 'Rows': 0, 'Cols': 0, 'Status': 'MISSING'})

df_validation = pd.DataFrame(results)
print(df_validation.to_string(index=False))
print()

total = len(results)
passed = sum(1 for r in results if r['Status'] == 'PASS')
print(f"\nValidation: {passed}/{total} files passed all checks")

assert all_passed, "Some files failed validation — check output above"
print("All files validated successfully!")

DATA VALIDATION
                  File  Exists  Rows  Cols Status
  elap733 missed games    True 11531     6   PASS
   elap733 injury txns    True 14135     6   PASS
  elap733 player stats    True 19786    32   PASS
elap733 team schedules    True 28240     9   PASS
     player_stats_2010    True   452    69   PASS
     player_stats_2011    True   478    69   PASS
     player_stats_2012    True   469    69   PASS
     player_stats_2013    True   482    69   PASS
     player_stats_2014    True   492    69   PASS
     player_stats_2015    True   476    69   PASS
     player_stats_2016    True   486    69   PASS
     player_stats_2017    True   540    69   PASS
     player_stats_2018    True   530    69   PASS
     player_stats_2019    True   529    69   PASS
       player_bio_2010    True   452    25   PASS
       player_bio_2011    True   478    25   PASS
       player_bio_2012    True   469    25   PASS
       player_bio_2013    True   482    25   PASS
       player_bio_2014    True   4

---
## Summary

### Data Collected

| # | Source | Directory | Files | Description |
|---|--------|-----------|-------|-------------|
| 1 | **elap733** | `data/raw/elap733/` | 4 CSVs | Injury transactions, missed games (target variable), historical player stats, team schedules |
| 2 | **nba_api** — Player Stats | `data/raw/nba_api/` | 10 CSVs | Per-game player statistics (69 cols) for each season 2010-2019 |
| 3 | **nba_api** — Player Bio | `data/raw/nba_api/` | 10 CSVs | Player demographics (height, weight, draft info, 25 cols) for each season 2010-2019 |
| 4 | **nba_api** — Tracking Stats | `data/raw/nba_api/` | 7 CSVs | Player speed & distance tracking (18 cols) for seasons 2013-2019 |
| 5 | **nba_api** — Team Schedules | `data/raw/nba_api/` | 10 CSVs | Game logs for all 30 teams × 10 seasons (30 cols each) |
| 6 | **nba_api** — Hustle Stats | `data/raw/nba_api/` | 1 CSV | Contested shots, deflections, loose balls (29 cols) for seasons 2016-2019 |

**Total: 42 raw CSV files** covering the 2010-2019 NBA seasons.

In [20]:
# Final file inventory
print("=" * 60)
print("ALL COLLECTED DATA FILES")
print("=" * 60)

print("\nelap733 data:")
for f in sorted(os.listdir(f"../{RAW_ELAP_DIR}")):
    if f.endswith('.csv'):
        size = os.path.getsize(f"../{RAW_ELAP_DIR}/{f}") / 1024
        print(f"  {f}: {size:.1f} KB")

print("\nnba_api data:")
for f in sorted(os.listdir(f"../{RAW_NBA_API_DIR}")):
    if f.endswith('.csv'):
        size = os.path.getsize(f"../{RAW_NBA_API_DIR}/{f}") / 1024
        print(f"  {f}: {size:.1f} KB")

ALL COLLECTED DATA FILES

elap733 data:
  injury_transactions_2010_2019.csv: 873.8 KB
  missed_games_2010_2019.csv: 739.8 KB
  player_stats_1994_2019.csv: 2778.2 KB
  team_schedules_2010_2020.csv: 1918.7 KB

nba_api data:
  hustle_stats_2016_2019.csv: 276.2 KB
  player_bio_2010.csv: 62.3 KB
  player_bio_2011.csv: 66.0 KB
  player_bio_2012.csv: 64.8 KB
  player_bio_2013.csv: 66.8 KB
  player_bio_2014.csv: 69.4 KB
  player_bio_2015.csv: 66.1 KB
  player_bio_2016.csv: 67.8 KB
  player_bio_2017.csv: 75.8 KB
  player_bio_2018.csv: 74.3 KB
  player_bio_2019.csv: 74.8 KB
  player_stats_2010.csv: 131.0 KB
  player_stats_2011.csv: 138.7 KB
  player_stats_2012.csv: 136.3 KB
  player_stats_2013.csv: 140.1 KB
  player_stats_2014.csv: 143.3 KB
  player_stats_2015.csv: 138.7 KB
  player_stats_2016.csv: 142.0 KB
  player_stats_2017.csv: 157.8 KB
  player_stats_2018.csv: 155.1 KB
  player_stats_2019.csv: 154.9 KB
  team_schedules_2010.csv: 340.3 KB
  team_schedules_2011.csv: 273.5 KB
  team_schedules_

---
## Handoff to Notebook 02 (Data Cleaning)

All raw data has been collected and saved. The next notebook (`02_data_cleaning.ipynb`) expects these files:

### Input files for NB02

**From `data/raw/elap733/`:**
- `missed_games_2010_2019.csv` — Target variable source (games missed per injury event)
- `injury_transactions_2010_2019.csv` — IRL transaction records

**From `data/raw/nba_api/`:**
- `player_stats_{2010..2019}.csv` (10 files) — Per-game player statistics
- `player_bio_{2010..2019}.csv` (10 files) — Player demographics
- `tracking_stats_{2013..2019}.csv` (7 files) — Speed/distance tracking data
- `team_schedules_{2010..2019}.csv` (10 files) — Team game logs
- `hustle_stats_2016_2019.csv` (1 file) — Hustle metrics

### What NB02 does with this data
1. Concatenates per-season CSVs into unified DataFrames
2. Standardizes player names across data sources
3. Handles missing values and data type conversions
4. Saves 7 cleaned CSVs to `data/processed/`